# 05 — Interactive Visualizations

## Purpose
This notebook produces all the **interactive Plotly charts** — primarily Sankey diagrams
and trend line charts. These are the 'presentation layer' visuals, useful for sharing findings
with the business team.

**What is a Sankey diagram?**
A Sankey diagram shows flows between categories. The width of each flow is proportional
to the quantity. In our case: branches flow into their top products, and the thickness
shows how much of each product was sold.

## What we build
1. Sankey: all branches → top 5 products (overall)
2. Sankey: all branches → top 5 products on cold days
3. Sankey: all branches → top 5 products on warm days
4. Sankey: day parts (morning/afternoon/night) → top products
5. Line charts: monthly trend for top 5 items per branch
6. Top 3 food groups per branch by month and day

## Input
`data/intermediate/datanomodifier.csv`

## Libraries
`pip install plotly`

## Run order
Run after `01_data_cleaning.ipynb`.

In [1]:
import os

# ─── CHANGE THIS FLAG ─────────────────────────────────────────────────
USE_GITHUB  = False   # True = read from GitHub (Colab), False = read from local clone
# ───────────────────────────────────────────────────────────────────────

GITHUB_BASE = "https://media.githubusercontent.com/media/DiegoLarrieta/PanemReto/main"

if USE_GITHUB:
    PROCESSED_DIR    = f"{GITHUB_BASE}/data/processed"
    WEATHER_DIR      = f"{GITHUB_BASE}/data/weather"
    RAW_DIR          = f"{GITHUB_BASE}/data/raw/Complete Data"
    INTERMEDIATE_DIR = None  # not in repo — run 00_data_pipeline.ipynb locally first
else:
    PROJECT_ROOT     = os.path.abspath(os.path.join(os.getcwd(), ".."))
    PROCESSED_DIR    = os.path.join(PROJECT_ROOT, "data", "processed")
    WEATHER_DIR      = os.path.join(PROJECT_ROOT, "data", "weather")
    RAW_DIR          = os.path.join(PROJECT_ROOT, "data", "raw", "Complete Data")
    INTERMEDIATE_DIR = os.path.join(PROJECT_ROOT, "data", "intermediate")

print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR)

INTERMEDIATE_DIR: /Users/diego/Documents/ChallengeAI/data/intermediate


## Setup — Load and clean data

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

# ── Load ──────────────────────────────────────────────────────────────
input_path = os.path.join(INTERMEDIATE_DIR, "datanomodifier.csv")
df = pd.read_csv(input_path, low_memory=False)
print(f"Loaded: {len(df):,} rows | {df.shape[1]} columns")

# ── Parse datetime columns ────────────────────────────────────────────
for col in ["operating_date", "closing_time", "captured_time"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# ── Fix is_modifier to boolean ────────────────────────────────────────
df["is_modifier"] = (
    df["is_modifier"].astype("string").str.strip().str.lower()
    .map({"true": True, "false": False}).fillna(False).astype(bool)
)

# ── Ensure numeric types ──────────────────────────────────────────────
df["quantity"]     = pd.to_numeric(df["quantity"],     errors="coerce").fillna(0)
df["total_ticket"] = pd.to_numeric(df["total_ticket"], errors="coerce").fillna(0)
df["tavg"]         = pd.to_numeric(df["tavg"],         errors="coerce")

# ── Remove beverages ──────────────────────────────────────────────────
df = df[~df["group"].isin(["CAFE Y BEBIDAS CALIENTES", "JUGOS Y BEBIDAS FRIAS"])].copy()

# ── Standardize branch names ──────────────────────────────────────────
df["sucursal"] = df["sucursal"].replace({
    "Panem - Hotel Kavia N"      : "Panem - Hotel Kavia",
    "Panem - Plaza QIN N"        : "Panem - Plaza QIN",
    "Panem - Hospital Zambrano N": "Panem - Hospital Zambrano",
    "Panem - La Carreta N"       : "Panem - Carreta",
})

# ── Fix product name typos ────────────────────────────────────────────
df["item"] = df["item"].replace({"SANDWITCH ENSALADA POLLO": "SANDWICH ENSALADA POLLO"})

# ── Temperature label ─────────────────────────────────────────────────
df["cold_or_warm"] = np.where(df["tavg"] >= 25, "warm", "cold")

# ── Create base ───────────────────────────────────────────────────────
base = df[
    (df["is_modifier"] == False) &
    (df["action"]      == "Venta") &
    (df["item"].notna())
].copy()

print(f"df shape:   {df.shape}")
print(f"base shape: {base.shape}")
print(f"Branches:   {sorted(base['sucursal'].dropna().unique())}")

Loaded: 2,579,572 rows | 50 columns
df shape:   (1439071, 50)
base shape: (1412688, 50)
Branches:   ['Panem - Carreta', 'Panem - Credi Club', 'Panem - Hospital Zambrano', 'Panem - Hotel Kavia', 'Panem - Plaza Nativa', 'Panem - Plaza QIN', 'Panem - Punto Valle']


## Helper — Sankey builder function

We define one reusable function instead of copy-pasting the chart code 4 times.
This follows the DRY principle (Don't Repeat Yourself).

In [3]:
def hex_to_rgba(color, alpha=0.4):
    """Convert a hex color to rgba string for Plotly link transparency."""
    h = color.lstrip("#")
    if len(h) == 6:
        r, g, b = (int(h[i:i+2], 16) for i in (0, 2, 4))
        return f"rgba({r},{g},{b},{alpha})"
    return f"rgba(120,120,120,{alpha})"


def build_sankey(df_input, group_col, item_col, qty_col, title, top_n=5):
    """
    Build a Plotly Sankey diagram: group_col (left) → top N items (right).

    Parameters
    ----------
    df_input  : DataFrame already filtered to the desired subset
    group_col : column whose unique values become the LEFT nodes (e.g. 'sucursal')
    item_col  : column whose values become the RIGHT nodes (e.g. 'item')
    qty_col   : numeric column to sum as flow width
    title     : chart title
    top_n     : how many items to show per group node
    """
    g = df_input.groupby([group_col, item_col], as_index=False)[qty_col].sum()

    # Rank items by total quantity (highest-selling on top)
    branch_totals = g.groupby(group_col)[qty_col].sum().sort_values(ascending=False)
    groups = branch_totals.index.astype(str).tolist()

    top = (
        g.sort_values([group_col, qty_col], ascending=[True, False])
         .groupby(group_col, group_keys=False)
         .head(top_n)
         .copy()
    )

    # Create unique item labels per group to avoid duplicate nodes
    top["item_node"] = top[item_col].astype(str) + " | " + top[group_col].astype(str)

    labels = [f"{g}" for g in groups] + top["item_node"].tolist()
    label_idx = {l: i for i, l in enumerate(labels)}

    colors = px.colors.qualitative.Plotly
    node_colors = [colors[i % len(colors)] for i in range(len(groups))] + \
                  ["lightgray"] * len(top)
    link_colors = [hex_to_rgba(colors[groups.index(str(r[group_col])) % len(colors)])
                   for _, r in top.iterrows()]

    fig = go.Figure(data=[go.Sankey(
        node=dict(pad=15, thickness=20, label=labels, color=node_colors),
        link=dict(
            source=[label_idx[str(r[group_col])] for _, r in top.iterrows()],
            target=[label_idx[r["item_node"]] for _, r in top.iterrows()],
            value=top[qty_col].tolist(),
            color=link_colors
        )
    )])
    fig.update_layout(title_text=title, font_size=11, height=600)
    fig.show()


## Chart 1 — Branch → Top 5 Products (Overall)

The big picture: which products does each branch sell the most of, in total?
The flow width represents total units sold across all years.

In [4]:
build_sankey(
    base, group_col="sucursal", item_col="item", qty_col="quantity",
    title="Panem — Branch → Top 5 Products (All Time)", top_n=5
)

## Chart 2 — Branch → Top 5 Products on Cold Days

Restricting to cold days (tavg < 25°C) shows which products dominate in cooler weather.
Compare this to the warm-day chart below — differences indicate temperature-sensitive products.

In [5]:
build_sankey(
    base[base["cold_or_warm"] == "cold"],
    group_col="sucursal", item_col="item", qty_col="quantity",
    title="Panem — Branch → Top 5 Products (COLD Days Only)", top_n=5
)

## Chart 3 — Branch → Top 5 Products on Warm Days

Same chart for warm days (tavg >= 25°C). Any product that appears here but not in the
cold chart is a seasonally driven item — its forecasting model will benefit
from having `tavg` or `cold_or_warm` as a feature.

In [6]:
build_sankey(
    base[base["cold_or_warm"] == "warm"],
    group_col="sucursal", item_col="item", qty_col="quantity",
    title="Panem — Branch → Top 5 Products (WARM Days Only)", top_n=5
)

## Chart 4 — Day Part → Top 10 Products

Instead of branches on the left, we now use `day_part` (morning / afternoon / night).
This shows which products dominate each part of the day across all branches combined.

In [7]:
build_sankey(
    base, group_col="day_part", item_col="item", qty_col="quantity",
    title="Panem — Day Part → Top 10 Products (All Branches)", top_n=10
)

## Chart 5 — Monthly trend line: Top 5 items per branch

This line chart shows how the monthly sales quantity of each branch's top 5 items
evolved over time. Look for:
- Consistent items that hold their rank over years (stable demand → easier to forecast)
- Items with seasonal spikes (need seasonal model components)
- Items declining over time (may drop out of the top 5)

One chart is produced per branch.

In [8]:
MONTH_ORDER = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']

for branch in sorted(base["sucursal"].dropna().unique()):
    d = base[base["sucursal"] == branch].copy()
    d["month_num"] = d["operating_date"].dt.month
    d["month"]     = pd.Categorical(d["operating_date"].dt.month_name(), categories=MONTH_ORDER, ordered=True)

    # Identify top 5 items by total volume for this branch
    top5 = (
        d.groupby("item")["quantity"].sum()
         .sort_values(ascending=False).head(5).index.tolist()
    )

    monthly = (
        d[d["item"].isin(top5)]
        .groupby(["month_num", "month", "item"], observed=True, as_index=False)["quantity"]
        .sum()
        .sort_values(["month_num", "item"])
    )

    if monthly.empty:
        continue

    fig = px.line(
        monthly, x="month", y="quantity", color="item",
        title=f"{branch} — Monthly Quantity Trend: Top 5 Items",
        markers=True
    )
    fig.update_layout(height=450, legend_title="Product")
    fig.show()

## Chart 6 — Top 3 food groups per branch, by month and day

Instead of individual products, we aggregate by **food group** (e.g., COMIDAS, PAN DULCE).
For each combination of (branch, month, day of week) we identify which 3 food groups
sell the most.

This chart reveals structural patterns — which category anchors each branch's sales.

In [9]:
month_names = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
day_order_list = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

d = base.copy()
d["operating_date"] = pd.to_datetime(d["operating_date"], errors="coerce")
d = d.dropna(subset=["operating_date", "sucursal", "group"])
d = d[d["quantity"] > 0]
d["month_num"] = d["operating_date"].dt.month
d["month"]    = pd.Categorical(d["operating_date"].dt.month_name(), categories=month_names, ordered=True)
d["weekday"]  = pd.Categorical(d["operating_date"].dt.day_name(), categories=day_order_list, ordered=True)

agg = (
    d.groupby(["sucursal", "month_num", "month", "weekday", "group"], observed=True, as_index=False)
     .agg(total_qty=("quantity", "sum"))
)
agg["rank"] = agg.groupby(["sucursal", "month_num", "weekday"])["total_qty"].rank(method="first", ascending=False)

top3 = agg[agg["rank"] <= 3].copy()
top3["rank"] = top3["rank"].astype(int)

for branch in sorted(top3["sucursal"].dropna().unique()):
    b = top3[top3["sucursal"] == branch].copy()
    for r in [1, 2, 3]:
        dominant = (
            b[b["rank"] == r]
            .groupby(["sucursal", "weekday"], observed=True)["group"]
            .agg(lambda s: s.value_counts().index[0] if len(s) else None)
            .reset_index(name=f"top{r}_group")
        )
    print(f"\n{branch}")
    display(dominant.drop(columns="sucursal"))


Panem - Carreta


/var/folders/h4/srr2qc116dg83xvd0rkbfh6c0000gn/T/ipykernel_63582/1275966641.py:17: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,weekday,top3_group
0,Monday,DESAYUNOS
1,Tuesday,DESAYUNOS
2,Wednesday,DESAYUNOS
3,Thursday,DESAYUNOS
4,Friday,DESAYUNOS
5,Saturday,COMIDAS



Panem - Credi Club


,weekday,top3_group
0,Monday,COMIDAS
1,Tuesday,COMIDAS
2,Wednesday,COMIDAS
3,Thursday,COMIDAS
4,Friday,UBER PAN DULCE
5,Saturday,UBER PAN DULCE
6,Sunday,UBER DESAYUNOS


...

,weekday,top3_group
0,Monday,REPOSTERIA
1,Tuesday,REPOSTERIA
2,Wednesday,REPOSTERIA
3,Thursday,REPOSTERIA
4,Friday,REPOSTERIA
5,Saturday,REPOSTERIA
6,Sunday,REPOSTERIA



Panem - Plaza QIN


,weekday,top3_group
0,Monday,COMIDAS
1,Tuesday,COMIDAS
2,Wednesday,DESAYUNOS
3,Thursday,COMIDAS
4,Friday,COMIDAS
5,Saturday,REPOSTERIA
6,Sunday,REPOSTERIA



Panem - Punto Valle


,weekday,top3_group
0,Monday,COMIDAS
1,Tuesday,COMIDAS
2,Wednesday,COMIDAS
3,Thursday,COMIDAS
4,Friday,COMIDAS
5,Saturday,REPOSTERIA
6,Sunday,REPOSTERIA


## Summary

All interactive charts are viewable inline in Jupyter. Key things to note:
- The Sankey diagrams are interactive — hover for exact quantities
- Cold vs warm Sankey differences highlight temperature-sensitive products for the model
- Monthly trend lines show which top-5 items are stable vs seasonal

**Next step:** `06_feature_engineering.ipynb` — transform the data into ML-ready datasets.